In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Flatten, Dense, Dropout, Lambda
from keras import backend as K
import numpy as np
from tensorflow.keras.datasets import mnist
import os
from PIL import Image
from tensorflow.keras.callbacks import ModelCheckpoint

In [ ]:
print(tf.__version__)

2.17.1


In [ ]:
!pip uninstall -y tensorflow


Found existing installation: tensorflow 2.17.1
Uninstalling tensorflow-2.17.1:
  Successfully uninstalled tensorflow-2.17.1


In [ ]:
!pip install tensorflow==2.8.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 497.6/497.6 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 462.5/462.5 kB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 63.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.6/42.6 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.8/5.8 MB 84.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 106.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 781.3/781.3 kB 53.1 MB/s eta 0:00:00
  Attempting uninstall: keras
    Found existing installation: keras 3.5.0
    Uninstalling keras-3.5.0:
      Successfully uninstalled keras-3.5.0
  Attempting uninstall: tensorboard-data-server
    Found existing installation: tensorboard-data-server 0.7.2
    Uninstalling tensorboard-data-server-0.7.2:
      Successfully uninstalled tensorboard-data-server-0.7.2
  Attempting uninstall: google-auth-oauthlib
    Found existing installation: go

In [ ]:
!pip install protobuf==3.20.3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 48.1 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 4.25.5
    Uninstalling protobuf-4.25.5:
      Successfully uninstalled protobuf-4.25.5
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grpcio-status 1.62.3 requires protobuf>=4.21.6, but you have protobuf 3.20.3 which is incompatible.
pandas-gbq 0.24.0 requires google-auth-oauthlib>=0.7.0, but you have google-auth-oauthlib 0.4.6 which is incompatible.
tf-keras 2.17.0 requires tensorflow<2.18,>=2.17, but you have tensorflow 2.8.0 which is incompatible.


In [ ]:
def siamese_model(input_shape):
    input = tf.keras.layers.Input(shape=input_shape)
    #input shape 72x64 (heightxwidth) output 72x64x1 (heightxwidthx1channel)
    x = tf.keras.layers.Conv2D(32, (5, 5), activation='relu')(input)
    x = tf.keras.layers.Conv2D(64, (3, 3), activation='relu')(x)                                                                                                                                                                 # Example of a simple convolutional branch                                                                              x = tf.keras.layers.Conv2D(32, (5, 5), activation='relu')(input)                                                        x = tf.keras.layers.Conv2D(64, (3, 3), activation='relu')(x)
    #depthise convolution across every channel
    x = tf.keras.layers.MaxPooling2D((2, 2))(x)
    #pools everything in a 2x2 window reducing the size by 2
    #output is 35x31x64
    x = tf.keras.layers.Conv2D(64, (1, 1), activation='relu')(x)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    #returns a 1x64 vector

    return tf.keras.Model(inputs=input, outputs=x)

In [ ]:
# Define input shapes
input_shape = (244, 244, 1)

# Create two identical models for each input in the pair
base_model = siamese_model(input_shape)
base_model.summary()

input_a = tf.keras.layers.Input(shape=input_shape)
input_b = tf.keras.layers.Input(shape=input_shape)

# Apply the same model to both inputs
processed_a = base_model(input_a)
processed_b = base_model(input_b)

# Compute the distance between the two embeddings
distance = tf.keras.layers.Lambda(lambda tensors: tf.keras.backend.abs(tensors[0] - tensors[1]))([processed_a, processed_b])

# Binary classification layer (same or different)
output = tf.keras.layers.Dense(1, activation='sigmoid')(distance)

# Final model
model = tf.keras.Model([input_a, input_b], output)

model.summary()
# Compile the model with binary cross-entropy
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)  # Start with a lower learning rate
model.compile(loss='binary_crossentropy', optimizer=optimizer, metrics=['accuracy'])

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 244, 244, 1)]     0         
                                                                 
 conv2d (Conv2D)             (None, 240, 240, 32)      832       
                                                                 
 conv2d_1 (Conv2D)           (None, 238, 238, 64)      18496     
                                                                 
 max_pooling2d (MaxPooling2D  (None, 119, 119, 64)     0         
 )                                                               
                                                                 
 conv2d_2 (Conv2D)           (None, 119, 119, 64)      4160      
                                                                 
 global_average_pooling2d (G  (None, 64)               0         
 lobalAveragePooling2D)                                      

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Fingerprint Processing


In [ ]:
# Path to the directory containing your BMP images
image_dir = 'Fingerprints'

# Initialize lists to store image data and labels
X_train_full = []
y_train_full = []

# Finger mapping dictionary with separate labels for Ansen and Zianne
finger_mapping = {
    "Ansen": {
        "RThumb": 0,
        "RPointer": 1,
        "RMiddle": 2,
        "RRing": 3,
        "RPinky": 4,
        "LThumb": 5,
        "LPointer": 6,
        "LMiddle": 7,
        "LRing": 8,
        "LPinky": 9
    },
    "Zianne": {
        "RThumb": 10,
        "RPointer": 11,
        "RMiddle": 12,
        "RRing": 13,
        "RPinky": 14,
        "LThumb": 15,
        "LPointer": 16,
        "LMiddle": 17,
        "LRing": 18,
        "LPinky": 19
    }
}

# Loop through all images in the directory
for image_filename in os.listdir(image_dir):
    if image_filename.endswith('.jpg'):
        # Load the image using PIL, convert to grayscale
        img = Image.open(os.path.join(image_dir, image_filename)).convert('L')
        img = img.resize((244, 244))  # Resize as needed

        # Convert the image to a NumPy array and normalize pixel values
        img_array = np.array(img) / 255.0
        X_train_full.append(img_array)

        # Extract the name and finger label from the filename
        parts = image_filename.split('_')
        if len(parts) >= 3:  # Check if filename has expected parts
            person = parts[0]  # "Ansen" or "Zianne"
            finger_label = parts[1]  # Finger type (e.g., "RThumb")

                        # Check if the finger is from the left hand, and if so, mirror the image
            if finger_label.startswith('L'):
                img = img.transpose(Image.FLIP_LEFT_RIGHT)

            if person in finger_mapping and finger_label in finger_mapping[person]:
                # Map the finger to its class label for that specific person
                label = finger_mapping[person][finger_label]
                y_train_full.append(label)
            else:
                print(f"Warning: Finger label '{finger_label}' or person '{person}' not recognized in {image_filename}")


# Convert lists to NumPy arrays
X_train_full = np.array(X_train_full)
y_train_full = np.array(y_train_full)

# Check the shape of the full dataset
print("Full dataset shape:", X_train_full.shape, y_train_full.shape)

# Randomly select 80 images for the training set
train_indices = np.random.choice(len(X_train_full), size=400, replace=False)
X_train = X_train_full[train_indices]
y_train = y_train_full[train_indices]

# Randomly select 30 images for the testing set
test_indices = np.random.choice(len(X_train_full), size=100, replace=False)

X_test = X_train_full[test_indices]
y_test = y_train_full[test_indices]

# Check the shape of the selected datasets
print("Training set shape:", X_train.shape, y_train.shape)
print("Testing set shape:", X_test.shape, y_test.shape)

Full dataset shape: (529, 122, 122) (529,)
Training set shape: (400, 122, 122) (400,)
Testing set shape: (100, 122, 122) (100,)


In [ ]:
def make_paired_dataset(X, y, num_pairs=10000):
    X_pairs, y_pairs = [], []

    # Create a list of indices based on the length of the dataset
    indices = np.arange(len(X))

    for _ in range(num_pairs):
        # Randomly select two indices from the dataset
        idx_A, idx_B = np.random.choice(indices, size=2, replace=False)

        # Get the images and labels using the randomly selected indices
        img_A, label_A = X[idx_A], y[idx_A]
        img_B, label_B = X[idx_B], y[idx_B]

        # Label is 1 if the pair has the same label, otherwise 0
        new_label = int(label_A == label_B)

        X_pairs.append([img_A, img_B])
        y_pairs.append(new_label)

    # Convert lists to NumPy arrays
    X_pairs = np.array(X_pairs)
    y_pairs = np.array(y_pairs)

    return X_pairs, y_pairs

In [ ]:
# Number of pairs you want to create (adjust based on your memory and model needs)
num_pairs = 30000  # You can modify this to any desired number

# Create the paired dataset using the function
X_train_pairs, y_train_pairs = make_paired_dataset(X_train, y_train, num_pairs=num_pairs)

# Check the shape of the paired data
print(X_train_pairs.shape)  # (num_pairs, 2, H, W) for grayscale images
print(y_train_pairs.shape)  # (num_pairs,)

num_pairs = 1000  # You can modify this to any desired number

# Create the paired dataset using the function
X_test_pairs, y_test_pairs = make_paired_dataset(X_test, y_test, num_pairs=num_pairs)

# Check the shape of the paired data
print(X_test_pairs.shape)  # (num_pairs, 2, H, W) for grayscale images
print(y_test_pairs.shape)  # (num_pairs,)

# Separate the pairs into two sets of images (for training and testing)
img_A_train = X_train_pairs[:, 0]  # First image in the pair
img_B_train = X_train_pairs[:, 1]  # Second image in the pair
img_A_test = X_test_pairs[:, 0]  # First image in the test pair
img_B_test = X_test_pairs[:, 1]  # Second image in the test pair

# Reshape if necessary (e.g., add the channel dimension for grayscale images)
img_A_train = img_A_train.reshape((-1, 244, 244, 1))
img_B_train = img_B_train.reshape((-1, 244, 244, 1))
img_A_test = img_A_test.reshape((-1, 244, 244, 1))
img_B_test = img_B_test.reshape((-1, 244, 244, 1))

(30000, 2, 122, 122)
(30000,)
(1000, 2, 122, 122)
(1000,)


# Training

In [ ]:
model.fit(
    [img_A_train, img_B_train],  # Input pairs (two sets of images)
    y_train_pairs,  # Labels (1 if same class, 0 if different class)
    validation_data=([img_A_test, img_B_test], y_test_pairs),  # Validation data
    epochs=80,
    batch_size=64
)

Epoch 1/65
469/469 [==============================] - 810s 2s/step - loss: 0.1928 - accuracy: 0.9503 - val_loss: 0.1673 - val_accuracy: 0.9380
Epoch 2/65
469/469 [==============================] - 804s 2s/step - loss: 0.1560 - accuracy: 0.9503 - val_loss: 0.1658 - val_accuracy: 0.9380
Epoch 3/65
469/469 [==============================] - 793s 2s/step - loss: 0.1494 - accuracy: 0.9503 - val_loss: 0.1571 - val_accuracy: 0.9380
Epoch 4/65
469/469 [==============================] - 792s 2s/step - loss: 0.1435 - accuracy: 0.9503 - val_loss: 0.1521 - val_accuracy: 0.9380
Epoch 5/65
469/469 [==============================] - 782s 2s/step - loss: 0.1387 - accuracy: 0.9503 - val_loss: 0.1444 - val_accuracy: 0.9380
Epoch 6/65
469/469 [==============================] - 782s 2s/step - loss: 0.1351 - accuracy: 0.9503 - val_loss: 0.1407 - val_accuracy: 0.9380
Epoch 7/65
469/469 [==============================] - 788s 2s/step - loss: 0.1299 - accuracy: 0.9503 - val_loss: 0.1381 - val_accuracy: 0.9380

In [ ]:
feature_extractor_model = tf.keras.Model(inputs=base_model.input, outputs=base_model.output)

# Save the model to disk
feature_extractor_model.save("ffe_model_122_20c_2.h5")

In [ ]:
feature_extractor_model = tf.keras.models.load_model("ffe_model_122_20c_2.h5")

single_image = X_train_full[13]
single_image = single_image.reshape(1, 122, 122, 1)

feature_extractor_model = tf.keras.models.load_model("ffe_model_122_20c_2.h5")

# Use the feature extractor model to predict the feature vector
feature_vector = feature_extractor_model.predict(single_image)

print(y_train_full)
# Print the feature vector
print("class :", y_train_full[13])
print("Feature Vector:", feature_vector)

for i in range(100, 200):

    compare_image = X_train_full[i]
    compare_image = compare_image.reshape(1, 122, 122, 1)
# Quantize and store AI models

# Use the feature extractor model to predict the feature vector
    feature_vector_compare = feature_extractor_model.predict(compare_image)

    from scipy.spatial import distance
    euclidean_distance = np.linalg.norm(feature_vector - feature_vector_compare)
    print(f"Euclidean Distance: {euclidean_distance} :", y_train_full[13], "vs ", y_train_full[i])

[12 18 13  6  8  4 13  5  6  1  6  1  2 19 19  8 15 11 18 12 11 16 13 10
 11  6 15 18 10 18  8  2  2 17  2  8  9 19  5  5  0  1  3 16 13  3  7  4
 17  1  7 13  3 11  0 12  8 19  0  1  2 11 14  8 10  3  0 13 13  5 10  5
  7 17 19 10 12 14 19 12 12  4  7  0  4  6 17  0 16 11  1 13  4 10 19 16
  9  5 11  4  6  8  7 14  5 18 17 10  6 12 11 14 10  6  7  4 10 18  6  5
 15  1 14  3  1  0  4 12  0  8 10 10  1 13 18  6  7 13 12 11  2 10 15  3
  3 15  8 13 16 18 10 17 17  1 12  3  1  2  5 16 18  8  6 15  4 10 19 19
 12  7  7 14 17  6 15 18 15  0 18 17 14  7 10 12  8  3 18  4 15  0 16  3
 11 16 16  6 11  0  0 14 15 17  2 12 11 15  3  8 16  1 16  3 11 11  7  3
  4 18  5  6  5 17 10  5  7 12  1  6 11  1 17 15 12 13 10 16 18  4  8  0
  0 13 15 16 13  3  2 16 11  2 11  5 15  0 19  4 11 10 12 12 12 19  5 15
  1  5  8 16  7 13  6 19 10 10 11 13 11  1  5  4 14  2 11 15  5 14  2  2
  9  6 18 16 19  3 18  6 11 12  4  1  9 15  4  0 10 15 19 11  8 13  9 17
 10  6 16 17  8 10 15  8 18  0 12 19  3  1 13  2 16